# 面试问题：LLM PPO 中怎样从 Token-level KL 和 GAE 得到带 mask 的 clipped policy loss？

        ## 可直接复述的回答主线

        1. LLM PPO 的样本单位是 response Token，序列奖励只在终止位置注入，再通过 GAE 向前分配信用。
2. 每个有效 Token 都要计算当前策略与参考策略的 KL 惩罚，padding 位置必须完全屏蔽。
3. 朴素地把序列奖励平均复制到每个 Token 无法表达位置价值和终止边界。
4. GAE 使用 value、下一步 value、终止 mask、gamma 和 lambda 递推 advantage。
5. 策略更新用新旧概率比和 clip 约束，监控 KL、clip fraction 与有效 Token 数。
6. 教学张量只能解释目标函数，真实 RLHF 还需要 rollout、value 拟合、分布式训练与奖励模型治理。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例是六条中文客服回复，每条保留 prompt、可读 response、人工或规则奖励和有效 Token 长度。对齐张量为离线固定小样本，模拟真实 rollout 的 old/current/reference log probability、value 和 padding mask，不含用户隐私。

In [1]:
import torch  # 使用基础张量运算手写 Token 级 PPO 目标。
torch.manual_seed(22)  # 固定教学实验随机状态以保证保存输出稳定。
samples = [{"id": "ppo-01", "prompt": "退款多久到账？", "response": "已受理，预计三个工作日到账。", "score": 1.0, "length": 4}, {"id": "ppo-02", "prompt": "如何补开发票？", "response": "请在订单页申请电子发票。", "score": 0.7, "length": 3}, {"id": "ppo-03", "prompt": "包裹显示异常。", "response": "建议等待。", "score": -0.2, "length": 2}, {"id": "ppo-04", "prompt": "会员如何取消？", "response": "可在设置页关闭自动续费。", "score": 0.9, "length": 4}, {"id": "ppo-05", "prompt": "能否提供密码？", "response": "不能索取密码，我可以帮助重置。", "score": 1.0, "length": 4}, {"id": "ppo-06", "prompt": "订单可以改地址吗？", "response": "发货前可在订单详情修改。", "score": 0.6, "length": 3}]  # 定义六条有语义的离线对齐样本。
max_tokens = 4  # 设定小批次 response 的统一 padding 长度。
mask = torch.tensor([[1 if position < sample["length"] else 0 for position in range(max_tokens)] for sample in samples], dtype=torch.float64)  # 构造每条 response 的有效 Token mask。
scores = torch.tensor([sample["score"] for sample in samples], dtype=torch.float64)  # 把序列奖励转换为批次张量。
print("教学实验输入：六条客服 PPO rollout")  # 标记下方为离线样本。
for sample in samples:  # 逐条展示 prompt、response、奖励和长度。
    print(f"{sample['id']} reward={sample['score']:>4.1f} len={sample['length']} | {sample['prompt']} -> {sample['response']}")  # 输出当前可读轨迹。
print("mask shape=", tuple(mask.shape), "有效Token=", int(mask.sum().item()))  # 展示张量形状和真实训练分母。

教学实验输入：六条客服 PPO rollout
ppo-01 reward= 1.0 len=4 | 退款多久到账？ -> 已受理，预计三个工作日到账。
ppo-02 reward= 0.7 len=3 | 如何补开发票？ -> 请在订单页申请电子发票。
ppo-03 reward=-0.2 len=2 | 包裹显示异常。 -> 建议等待。
ppo-04 reward= 0.9 len=4 | 会员如何取消？ -> 可在设置页关闭自动续费。
ppo-05 reward= 1.0 len=4 | 能否提供密码？ -> 不能索取密码，我可以帮助重置。
ppo-06 reward= 0.6 len=3 | 订单可以改地址吗？ -> 发货前可在订单详情修改。
mask shape= (6, 4) 有效Token= 20


## 2. Baseline / 基线：把中心化序列奖励复制到所有 Token

这个基线不使用 value，也不区分终止位置；它能给出正负方向，却无法解释哪个 Token 导致奖励，也会让短回复和长回复信用分配相同。

In [2]:
centered_scores = scores - scores.mean()  # 对批次序列奖励做简单中心化。
baseline_advantage = centered_scores[:, None].repeat(1, max_tokens) * mask  # 把同一序列优势复制到所有有效 Token。
print("Baseline 序列奖励复制结果")  # 标记当前输出属于朴素信用分配。
print("样本      reward  centered  每个有效Token优势")  # 输出基线表头。
for index, sample in enumerate(samples):  # 逐条展示有效 Token 上的相同优势。
    values = baseline_advantage[index][mask[index].bool()].tolist()  # 读取当前 response 的有效优势。
    print(f"{sample['id']:<9} {sample['score']:>6.2f} {centered_scores[index].item():>9.3f} {['%.3f' % value for value in values]}")  # 输出当前样本的基线信用。

Baseline 序列奖励复制结果
样本      reward  centered  每个有效Token优势
ppo-01      1.00     0.333 ['0.333', '0.333', '0.333', '0.333']
ppo-02      0.70     0.033 ['0.033', '0.033', '0.033']
ppo-03     -0.20    -0.867 ['-0.867', '-0.867']
ppo-04      0.90     0.233 ['0.233', '0.233', '0.233', '0.233']
ppo-05      1.00     0.333 ['0.333', '0.333', '0.333', '0.333']
ppo-06      0.60    -0.067 ['-0.067', '-0.067', '-0.067']


## 3. 底层实现：Token KL、终止奖励与 GAE 递推

固定张量模拟一次 rollout。KL 惩罚放在每个有效 Token，序列 score 只加到最后有效 Token；GAE 从右向左递推，并用 next mask 阻断 padding。

In [3]:
old_logp = torch.tensor([[-1.20, -0.90, -1.10, -0.80], [-1.00, -1.30, -0.70, 0.00], [-0.80, -1.10, 0.00, 0.00], [-1.40, -1.00, -0.90, -0.70], [-1.10, -0.80, -1.20, -0.90], [-0.90, -1.00, -0.80, 0.00]], dtype=torch.float64)  # 保存 rollout 时旧策略的 Token log probability。
current_logp = old_logp + torch.tensor([[0.05, -0.02, 0.08, 0.03], [0.02, 0.07, -0.04, 0.00], [-0.03, 0.04, 0.00, 0.00], [0.09, -0.01, 0.05, 0.02], [0.04, 0.06, -0.02, 0.07], [-0.05, 0.03, 0.02, 0.00]], dtype=torch.float64)  # 构造当前策略相对旧策略的小幅变化。
reference_logp = old_logp - 0.06 * mask  # 构造冻结参考策略用于 Token KL 代理。
values = torch.tensor([[0.30, 0.40, 0.50, 0.55], [0.20, 0.35, 0.45, 0.00], [-0.10, 0.05, 0.00, 0.00], [0.25, 0.38, 0.52, 0.62], [0.40, 0.50, 0.58, 0.70], [0.15, 0.28, 0.40, 0.00]], dtype=torch.float64) * mask  # 提供 rollout value 预测并清零 padding。
beta = 0.08  # 设定参考策略 KL 惩罚系数。
token_kl = (current_logp - reference_logp) * mask  # 计算每个有效 Token 的采样 KL 代理。
token_rewards = -beta * token_kl  # 把 KL 作为每步负奖励。
for row, sample in enumerate(samples):  # 把序列 score 注入每条 response 的终止 Token。
    terminal = sample["length"] - 1  # 找到当前 response 最后有效位置。
    token_rewards[row, terminal] += scores[row]  # 在终止位置加入任务奖励。
def compute_gae(rewards, value_predictions, valid_mask, gamma, lam):  # 从右向左手写带终止 mask 的 GAE。
    advantages = torch.zeros_like(rewards)  # 初始化每个 Token 的 advantage。
    next_advantage = torch.zeros(rewards.shape[0], dtype=rewards.dtype)  # 初始化每条序列的后继 GAE。
    for position in range(rewards.shape[1] - 1, -1, -1):  # 从最后一个 padding 槽向前递推。
        has_next = valid_mask[:, position + 1] if position + 1 < rewards.shape[1] else torch.zeros(rewards.shape[0], dtype=rewards.dtype)  # 判断下一位置是否仍属于同一 response。
        next_value = value_predictions[:, position + 1] if position + 1 < rewards.shape[1] else torch.zeros(rewards.shape[0], dtype=rewards.dtype)  # 读取下一有效位置的 value。
        delta = rewards[:, position] + gamma * next_value * has_next - value_predictions[:, position]  # 计算当前 Token 的时序差分残差。
        next_advantage = delta + gamma * lam * has_next * next_advantage  # 把后继信用折扣回当前 Token。
        advantages[:, position] = next_advantage * valid_mask[:, position]  # 只在有效 Token 上提交 advantage。
    return advantages  # 返回与 response 张量同形的 Token advantage。
advantages = compute_gae(token_rewards, values, mask, gamma=0.99, lam=0.95)  # 对六条 rollout 计算 Token 级 GAE。
ratios = torch.exp(current_logp - old_logp)  # 计算当前策略相对 rollout 策略的概率比。
clipped_ratios = torch.clamp(ratios, 0.8, 1.2)  # 对概率比应用 PPO clip 区间。
surrogate = torch.minimum(ratios * advantages, clipped_ratios * advantages)  # 按优势符号选择保守 surrogate。
policy_loss = -(surrogate * mask).sum() / mask.sum()  # 仅以有效 Token 为分母计算策略损失。
print("ppo-01 Token 级中间量")  # 选择一条可读样本展示 KL、奖励和 GAE。
print("位置  mask    KL      reward     value      GAE     ratio")  # 输出 Token 过程表头。
for position in range(max_tokens):  # 逐位置展示第一条 response 的核心张量。
    print(f"{position:>2} {mask[0, position].item():>5.0f} {token_kl[0, position].item():>7.3f} {token_rewards[0, position].item():>10.3f} {values[0, position].item():>9.3f} {advantages[0, position].item():>8.3f} {ratios[0, position].item():>8.3f}")  # 输出当前 Token 的 PPO 信号。

ppo-01 Token 级中间量
位置  mask    KL      reward     value      GAE     ratio
 0     1   0.110     -0.009     0.300    0.571    1.051
 1     1   0.040     -0.003     0.400    0.515    0.980
 2     1   0.140     -0.011     0.500    0.450    1.083
 3     1   0.090      0.993     0.550    0.443    1.030


## 4. 结果表与结果解读

下表比较序列奖励复制与 GAE 的逐样本平均优势。GAE 会受 value、KL 和距离终点影响，因此同一 response 内不再是常数。

In [4]:
print("样本      Baseline均值  GAE均值  首Token GAE  末Token GAE")  # 输出两种信用分配的同指标对照。
for index, sample in enumerate(samples):  # 逐条 response 汇总有效 Token 优势。
    valid = mask[index].bool()  # 取得当前 response 的有效位置。
    baseline_mean = baseline_advantage[index][valid].mean().item()  # 计算基线平均优势。
    gae_mean = advantages[index][valid].mean().item()  # 计算 GAE 平均优势。
    first_gae = advantages[index, 0].item()  # 读取首 Token 被回传的信用。
    last_gae = advantages[index, sample["length"] - 1].item()  # 读取终止 Token 优势。
    print(f"{sample['id']:<9} {baseline_mean:>12.3f} {gae_mean:>8.3f} {first_gae:>11.3f} {last_gae:>11.3f}")  # 输出当前样本的信用分配对照。
clip_fraction = (((ratios < 0.8) | (ratios > 1.2)).to(torch.float64) * mask).sum() / mask.sum()  # 计算有效 Token 上的 clip 比例。
print(f"结果解读：policy_loss={policy_loss.item():.4f}，平均Token KL={(token_kl.sum() / mask.sum()).item():.4f}，clip_fraction={clip_fraction.item():.3f}")  # 展示策略更新的三个关键监控量。

样本      Baseline均值  GAE均值  首Token GAE  末Token GAE
ppo-01           0.333    0.495       0.571       0.443
ppo-02           0.033    0.336       0.440       0.248
ppo-03          -0.867   -0.177      -0.096      -0.258
ppo-04           0.233    0.402       0.540       0.274
ppo-05           0.333    0.395       0.486       0.290
ppo-06          -0.067    0.295       0.400       0.194
结果解读：policy_loss=-0.3457，平均Token KL=0.0850，clip_fraction=0.000


## 5. 失败案例与修正

如果直接对四列张量求均值，padding 也进入分母，短回复的梯度会被额外缩小。下面比较错误全矩阵均值与按有效 Token 归一化的损失。

In [5]:
naive_loss = -surrogate.mean()  # 模拟忽略 padding 分母的错误策略损失。
corrected_loss = policy_loss  # 读取按有效 Token mask 归一化的正确损失。
padding_fraction = 1.0 - mask.mean().item()  # 计算当前批次 padding 占比用于解释偏差。
print(f"错误行为：全矩阵均值 loss={naive_loss.item():.4f}，padding占比={padding_fraction:.1%}")  # 展示 padding 稀释后的目标。
print(f"修正行为：有效Token均值 loss={corrected_loss.item():.4f}，分母={int(mask.sum().item())}")  # 展示正确分母和修正结果。

错误行为：全矩阵均值 loss=-0.2881，padding占比=16.7%
修正行为：有效Token均值 loss=-0.3457，分母=20


## 6. 生产边界

本实验没有训练 value head，也没有 rollout 延迟、奖励模型偏差、分布式 minibatch、优势白化和拒绝采样。生产中还需按 prompt 长度分桶，并监控 KL、熵、clip fraction、value loss 和奖励黑客。

In [6]:
diagnostics = {"valid_tokens": int(mask.sum().item()), "mean_kl": float((token_kl.sum() / mask.sum()).item()), "clip_fraction": float(clip_fraction.item()), "policy_loss": float(policy_loss.item())}  # 汇总一次 PPO 更新必须可观测的低维指标。
print("生产监控快照：", {name: round(value, 4) if isinstance(value, float) else value for name, value in diagnostics.items()})  # 输出可审计的训练诊断。

生产监控快照： {'valid_tokens': 20, 'mean_kl': 0.085, 'clip_fraction': 0.0, 'policy_loss': -0.3457}


## 7. 最小回归测试

只验证样本规模、mask、padding 清零和目标有限性。

In [7]:
assert len(samples) >= 5  # 保证案例至少包含五条可读 rollout。
assert advantages.shape == mask.shape  # 保证 Token GAE 与 response mask 完全对齐。
assert torch.all(advantages[mask == 0] == 0)  # 保证 padding 位置没有泄漏优势。
assert torch.isfinite(policy_loss)  # 保证 clipped policy objective 数值有限。
assert not torch.isclose(naive_loss, corrected_loss)  # 保证失败案例确实展示 padding 分母偏差。